# NMT MarianMT — Spanish–English Translation (**Student Version**)

**Task:** Neural Machine Translation (NMT), translating *Spanish → English* (or the reverse) using a pre-trained Transformer encoder–decoder.

**Model:** `Helsinki-NLP/opus-mt-es-en` (MarianMT) — encoder–decoder Transformer.

---

## Objetivos del notebook

En este notebook vas a:

1. Cargar un modelo de traducción automática basado en Transformer (MarianMT) usando `transformers`.
2. Traducir frases sencillas entre español e inglés.
3. Experimentar con distintas estrategias de decodificación:
   - Greedy (búsqueda codiciosa).
   - Beam search con distintos `num_beams`.
   - Sampling (top-k, top-p, temperatura).
4. Implementar una mini-métrica de evaluación basada en **solapamiento de n-gramas** (aprox. tipo BLEU muy simplificado).
5. Analizar algunos **errores típicos** de NMT (under-translation, over-translation, errores léxicos/estilísticos).

---

## Qué se asume que ya sabes

- Arquitectura Transformer encoder–decoder (Tema 3).
- Conceptos básicos de NMT (codificador, decodificador, atención).
- Nociones sobre estrategias de decodificación: greedy, beam search, sampling.
- Python y nociones de PyTorch.


## 1) (Opcional) Instalación / actualización de librerías

**Qué hacer:**  
Si estás en Google Colab o en un entorno donde no tienes `transformers`/`sentencepiece` instalados (o quieres actualizar), puedes usar la celda de abajo.

- Si ya tienes el entorno preparado (por ejemplo, viene dado por el profesor), puedes **saltar esta celda**.


In [ ]:
# TODO (opcional): instala o actualiza las librerías necesarias.
# Ejemplo orientativo (NO lo dejes comentado si lo necesitas):
# !pip install -q "transformers>=4.40.0" sentencepiece

## 2) Imports y configuración básica

**Qué hacer:**

1. Importar:
   - `torch`
   - `MarianMTModel`, `MarianTokenizer` de `transformers`.
   - Módulos básicos (`numpy`, `re`, `collections.Counter`).
2. Detectar si hay GPU disponible (`torch.cuda.is_available()`).
3. Fijar una variable `device` (`"cuda"` o `"cpu"`) para mover el modelo.
4. (Opcional) Fijar una semilla aleatoria para reproducibilidad.


In [ ]:
# TODO: importa las librerías necesarias.
# - torch
# - MarianMTModel, MarianTokenizer desde transformers
# - numpy, Counter, re

# TODO: define `device` = "cuda" si hay GPU disponible, en otro caso "cpu".
# TODO (opcional): fija semillas para torch y numpy.
# TODO: imprime el dispositivo que se está usando.


## 3) Carga del modelo MarianMT y del tokenizador

**Qué hacer:**

1. Escoger el par de idiomas:
   - Por defecto: `es → en`.
   - Opcionalmente, puedes probar `en → es` cambiando los códigos.
2. Construir el nombre del modelo MarianMT:
   - `"Helsinki-NLP/opus-mt-es-en"` para ES→EN.
   - `"Helsinki-NLP/opus-mt-en-es"` para EN→ES.
3. Cargar:
   - `tokenizer = MarianTokenizer.from_pretrained(model_name)`
   - `model = MarianMTModel.from_pretrained(model_name).to(device)`

**Importante:** Esta celda puede tardar unos segundos en ejecutarse la primera vez (descarga de pesos).


In [ ]:
# TODO: define `src_lang` y `tgt_lang` (por defecto "es" y "en").
# TODO: construye `model_name` con el patrón "Helsinki-NLP/opus-mt-{src_lang}-{tgt_lang}".
# TODO: carga el tokenizer y el modelo, y muévelos a `device`.
# TODO: imprime un mensaje confirmando que se ha cargado correctamente.


## 4) Explorando el tokenizador (subword units)

**Qué hacer:**

1. Elige una frase en español con alguna palabra algo técnica (por ejemplo, "metaheurísticas multiobjetivo en PLN").
2. Tokenízala con el `tokenizer` y mira:
   - Los `input_ids`.
   - Los tokens subword (`tokenizer.convert_ids_to_tokens`).
3. Vuelve a decodificar los IDs a texto para comprobar que la frase se reconstruye (más o menos).

**Objetivo:** Entender que MarianMT usa un vocabulario de subpalabras (SentencePiece/BPE) que puede partir palabras raras en varias piezas.


In [ ]:
# TODO: define una frase en español y tokenízala con el tokenizer.
# Pistas:
# - Usa tokenizer(frase, return_tensors="pt")
# - Imprime los input_ids
# - Convierte ids a subpalabras con tokenizer.convert_ids_to_tokens(...)
# - Decodifica de vuelta a texto con tokenizer.decode(..., skip_special_tokens=True)

# TODO: repite con otra frase que contenga palabras técnicas o poco frecuentes.


## 5) Mini-dataset artesanal de frases (ES→EN)

**Qué hacer:**

1. Define una lista de diccionarios con claves:
   - `"src"`: frase en español.
   - `"ref"`: traducción de referencia en inglés (lo más correcta posible).
2. Incluye al menos 4–5 pares de frases.
   - Intenta cubrir distintos tipos de contenido: frases simples, frases con terminología técnica, frases con negaciones, etc.
3. Imprime las frases para asegurarte de que se han definido bien.

Más adelante usarás estos pares para comparar la salida del modelo con las referencias.


In [ ]:
# TODO: define una lista `pairs` con diccionarios {"src": ..., "ref": ...}.

# Ejemplo de estructura:
# pairs = [
#   {"src": "Este es un curso de Procesamiento del Lenguaje Natural.",
#    "ref": "This is a Natural Language Processing course."},
#   ...
# ]

# TODO: recorre la lista e imprime cada SRC y REF para comprobar.


## 6) Función de traducción con parámetros de decodificación

**Qué hacer:**

Implementa una función `translate(sentences, ...)` que:

- Reciba una lista de frases (o una sola frase en string).
- Use el `tokenizer` para codificarlas (`padding=True`, `truncation=True`, `return_tensors="pt"`).
- Llame a `model.generate(...)` con parámetros como:
  - `num_beams` (para greedy/beam search).
  - `max_length`.
  - `do_sample`, `top_k`, `top_p`, `temperature` (para sampling).
- Decodifique el resultado con `tokenizer.batch_decode(..., skip_special_tokens=True)`.
- Devuelva una lista de strings con las traducciones generadas.

**Objetivo:** Tener una función reutilizable para los distintos experimentos de traducción.


In [ ]:
# TODO: implementa la función `translate` que envuelva tokenizer + model.generate + batch_decode.
# Requisitos mínimos:
# - Debe aceptar tanto una string como una lista de strings.
# - Debe mover los tensores al `device` seleccionado.
# - Debe permitir ajustar `num_beams`, `max_length`, `do_sample`, `top_k`, `top_p`, `temperature`.

# Sugerencia: define la firma aproximadamente así (puedes adaptarla):
# def translate(sentences, num_beams=1, max_length=64,
#               do_sample=False, top_k=None, top_p=None, temperature=1.0):
#     ...



## 7) Experimento 1 — Greedy decoding (num_beams = 1)

**Qué hacer:**

1. Extrae las frases origen (`"src"`) de tu lista `pairs`.
2. Traduce con `translate(..., num_beams=1)` (greedy).
3. Imprime, para cada par:
   - SRC
   - HYP (traducción generada por el modelo)
   - REF (referencia)

**Mini-ejercicio de análisis (para responder en markdown):**

- Identifica un ejemplo:
  - Donde la traducción sea **muy buena**.
  - Donde sea **aceptable** pero mejorable.
  - Donde sea **claramente incorrecta**.


In [ ]:
# TODO: extrae las frases origen y aplica traducción greedy.

# Pistas:
# - src_sentences = [ex["src"] for ex in pairs]
# - translations_greedy = translate(src_sentences, num_beams=1)

# TODO: recorre src_sentences, translations_greedy y las referencias, imprimiendo SRC / HYP / REF.


## 8) Experimento 2 — Beam search con distintos `num_beams`

**Qué hacer:**

1. Vuelve a traducir las mismas frases origen pero ahora con:
   - `num_beams = 4`
   - (Opcional) también con `num_beams = 8` si el tiempo lo permite.
2. Compara las salidas con las de greedy.

**Preguntas de reflexión (para responder en markdown):**

- ¿Aumentar `num_beams` mejora la calidad de la traducción?
- ¿Tienden las frases a ser más largas o más cortas?
- ¿Crees que el coste computacional aumenta de forma ligera o fuerte al aumentar `num_beams`?


In [ ]:
# TODO: traduce `src_sentences` con num_beams=4 (y opcionalmente 8).
# TODO: imprime para cada frase la salida greedy y la salida con beam search,
#        junto con la referencia, para poder compararlas.

# Sugerencia: guarda las traducciones con beam=4 en una lista `translations_beam4`
# para reutilizarlas más adelante.


## 9) Experimento 3 — Sampling (top-k, top-p, temperatura)

**Qué hacer:**

1. Elige una frase en español (por ejemplo, relacionada con IA o PLN).
2. Genera varias traducciones con:
   - `do_sample=True`
   - `top_k=50`, `top_p=0.9`
   - Distintas temperaturas (por ejemplo, 0.3, 0.7, 1.2).
3. Observa:

- ¿Cómo cambia la diversidad de las traducciones?
- ¿Hay traducciones que se vuelven “extrañas” o incoherentes para temperaturas altas?

**Objetivo:** Ver el *trade-off* entre diversidad y calidad cuando usamos sampling.


In [ ]:
# TODO: elige una frase en español y genera varias traducciones usando sampling.
# Pistas:
# - Usa num_beams=1, do_sample=True
# - Prueba distintos valores de `temperature` (p.ej. 0.3, 0.7, 1.2)
# - Mantén top_k=50 y top_p=0.9 (o ajusta si quieres experimentar).

# TODO: imprime las distintas traducciones y comenta tus observaciones en una celda de texto.


## 10) Mini-métrica de evaluación basada en n-gramas

**Qué hacer:**

1. Implementar una tokenización muy simple a minúsculas (separando por espacios y eliminando signos de puntuación).
2. Implementar una función `ngram_overlap(hyp, ref, n)` que:
   - Construya los n-gramas de `hyp` y `ref`.
   - Calcule el número de n-gramas en común.
   - Devuelva `overlap / total_ref_ngrams` (entre 0 y 1).
3. Calcular, para cada par:
   - Solapamiento de 1-gramas (unigramas).
   - Solapamiento de 2-gramas (bigramas).
4. Comparar las medias de overlap para:
   - Traducciones greedy.
   - Traducciones con beam search.

**Nota:** Esta métrica es muy simple y tiene muchas limitaciones (no captura sinónimos ni el orden flexible, etc.), pero sirve como aproximación intuitiva a BLEU.


In [ ]:
# TODO: implementa `tokenize_simple` y `ngram_overlap`.

# Requisitos:
# - tokenize_simple(text): pasa a minúsculas, elimina signos de puntuación básicos
#   y divide en tokens por espacios.
# - ngram_overlap(hyp, ref, n): calcula el solapamiento de n-gramas normalizado
#   respecto al número total de n-gramas de la referencia.



In [ ]:
# TODO: aplica la mini-métrica a greedy y beam, y compara resultados.

# Pistas:
# - `refs` puede ser la lista [ex["ref"] for ex in pairs].
# - Define una función auxiliar (opcional) que recorra hyps/refs y devuelva
#   una lista o array con (overlap_unigrama, overlap_bigrama) para cada par.
# - Calcula la media de cada columna para greedy y para beam=4.

# BONUS: imprime también los valores por frase individual para analizar casos concretos.


## 11) Análisis de errores típicos y conclusiones

**Qué hacer:**

1. Elige al menos **3 ejemplos** SRC–REF–HYP donde observes:
   - **Under-translation**: la traducción omite información importante.
   - **Over-translation**: la traducción añade información que no está en el original.
   - **Errores léxicos/estilísticos**: falsos amigos, registro extraño, etc.
2. Para cada ejemplo, indica:
   - Tipo de error.
   - Posible causa (modelo, datos, decodificación).
3. Relaciona lo observado con la teoría:
   - ¿Cómo pueden influir los hiperparámetros de decodificación (beam, longitud) en estos errores?
   - ¿Qué papel podrían tener los datos de entrenamiento?

---

### Conclusión

En este notebook has:

- Usado un modelo **encoder–decoder Transformer (MarianMT)** para traducción ES↔EN.
- Experimentado con **greedy**, **beam search** y **sampling**, observando su impacto en calidad y diversidad.
- Implementado una mini-métrica de evaluación basada en **n-gramas**.
- Analizado errores típicos de NMT y conectado estos fenómenos con la teoría del Tema 4 (NMT).

En clase puedes comparar tus resultados con los de otros compañeros/as y discutir qué configuraciones te parecen más razonables para un sistema NMT real.
